# Модуль-ноутбук: `data_prep`

Завантаження → очистка → **темпоральний спліт** → мітки (train-frozen) → фічі. `%run` підтягує `features`, `labels`, `config`.

**Залежності:** `%run` 01_features.ipynb, 02_labels.ipynb

In [ ]:
%run 01_features.ipynb
%run 02_labels.ipynb

In [ ]:
"""Load -> clean -> temporal split -> label (train-frozen) -> features.

Run standalone:  notebooks/03_data_prep.ipynb
  downloads the CSV if missing, writes data/processed/{train,test}.parquet and a
  data/processed/split_meta.json summary, and prints the split.

The public entry point is `prepare()`, which returns an in-memory bundle that train.py
consumes directly (no round-trip through disk required).
"""

import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd



# ----------------------------------------------------------------------------- IO

In [ ]:
def download_if_needed(path: Path = config.DATA_RAW) -> Path:
    """Fetch train.csv from HuggingFace if it isn't already on disk."""
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    try:  # preferred: huggingface_hub (handles auth/caching/resume)
        from huggingface_hub import hf_hub_download

        cached = hf_hub_download(
            repo_id=config.HF_REPO_ID, filename=config.HF_FILENAME, repo_type="dataset"
        )
        path.write_bytes(Path(cached).read_bytes())
    except Exception:  # fallback: plain HTTPS
        urllib.request.urlretrieve(config.HF_DIRECT_URL, path)  # noqa: S310
    return path

In [ ]:
def load_raw(path: Path = config.DATA_RAW) -> pd.DataFrame:
    return pd.read_csv(download_if_needed(path))


# ----------------------------------------------------------------------------- cleaning

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Drop unusable rows and the EDA-flagged dead columns. Does NOT touch features/labels."""
    df = df.copy()
    play = pd.to_numeric(df["play_count"], errors="coerce")
    keep = play.notna() & (play > 0) & df[config.CREATOR_COL].notna()
    df = df.loc[keep].reset_index(drop=True)
    df = df.drop(columns=[c for c in config.DROP_COLS if c in df.columns], errors="ignore")
    return df

In [ ]:
def temporal_split(df: pd.DataFrame, test_fraction: float = config.TEST_FRACTION):
    """Oldest (1-f) -> train, newest f -> test, by create_time. Respects drift.

    Implausible epochs (epoch≈0 garbage) sort to the oldest end -> land in train.
    """
    ct = pd.to_numeric(df[config.TIME_COL], errors="coerce").fillna(0)
    order = np.argsort(ct.values, kind="stable")
    df_sorted = df.iloc[order].reset_index(drop=True)
    ct_sorted = ct.values[order]
    n_test = max(1, int(round(len(df_sorted) * test_fraction)))

    # Cut on a TIMESTAMP value, not a position: test = the boundary epoch and everything
    # newer; train = everything strictly older. So contemporaneous (and possibly correlated)
    # posts never straddle the split and train.max() < test.min() strictly when times differ.
    boundary = ct_sorted[-n_test]
    test_mask = ct_sorted >= boundary
    if (~test_mask).sum() == 0:  # degenerate: all rows share one timestamp -> positional split
        test_mask = np.arange(len(df_sorted)) >= (len(df_sorted) - n_test)
    train_df = df_sorted.loc[~test_mask].reset_index(drop=True)
    test_df = df_sorted.loc[test_mask].reset_index(drop=True)
    return train_df, test_df


# ----------------------------------------------------------------------------- orchestration

In [ ]:
def _xy(df: pd.DataFrame, thresholds: dict):
    """Build (raw_df, X, y) with NaN-label rows dropped consistently."""
    y = labels.make_labels(df, thresholds)
    mask = y.notna()
    df2 = df.loc[mask].reset_index(drop=True)
    y2 = y.loc[mask].reset_index(drop=True).astype(int)
    X = features.engineer_features(df2)
    return df2, X, y2

In [ ]:
def prepare() -> dict:
    """Full deterministic pipeline -> in-memory bundle (the single source for train.py)."""
    raw = clean(load_raw())
    train_df, test_df = temporal_split(raw)

    # CRITICAL: label thresholds fit on TRAIN ONLY, then frozen and applied to TEST.
    thresholds = labels.fit_creator_thresholds(train_df)

    train_df, X_train, y_train = _xy(train_df, thresholds)
    test_df, X_test, y_test = _xy(test_df, thresholds)

    def _range(d):
        ct = pd.to_datetime(pd.to_numeric(d[config.TIME_COL], errors="coerce"),
                            unit="s", utc=True)
        ct = ct[ct.dt.year > 2010]  # ignore epoch garbage when describing the range
        return [str(ct.min().date()), str(ct.max().date())] if len(ct) else [None, None]

    split_meta = {
        "train_n": int(len(X_train)), "test_n": int(len(X_test)),
        "train_pos_rate": float(y_train.mean()), "test_pos_rate": float(y_test.mean()),
        "train_date_range": _range(train_df), "test_date_range": _range(test_df),
        "n_creators_train": int(train_df[config.CREATOR_COL].nunique()),
        "test_fraction": config.TEST_FRACTION,
        "label": "within-creator ER > train per-creator median ER",
        "timezone_assumption": config.ASSUME_TIMEZONE,
    }
    return {
        "train_df": train_df, "test_df": test_df,
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "thresholds": thresholds, "split_meta": split_meta,
    }

In [ ]:
def main() -> None:
    b = prepare()
    config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    pd.concat([b["X_train"], b["y_train"]], axis=1).to_parquet(
        config.DATA_PROCESSED / "train.parquet")
    pd.concat([b["X_test"], b["y_test"]], axis=1).to_parquet(
        config.DATA_PROCESSED / "test.parquet")
    (config.DATA_PROCESSED / "split_meta.json").write_text(
        json.dumps(b["split_meta"], indent=2), encoding="utf-8")
    m = b["split_meta"]
    print("Data prepared:")
    print(f"  train: {m['train_n']} rows  (pos rate {m['train_pos_rate']:.3f})  "
          f"{m['train_date_range']}")
    print(f"  test : {m['test_n']} rows  (pos rate {m['test_pos_rate']:.3f})  "
          f"{m['test_date_range']}")
    print(f"  features: {len(features.FEATURE_COLUMNS)}  |  creators(train): "
          f"{m['n_creators_train']}")
    print(f"  wrote {config.DATA_PROCESSED}/train.parquet, test.parquet, split_meta.json")


if __name__ == "__main__":
    main()

In [ ]:
from types import SimpleNamespace
data_prep = SimpleNamespace(
    download_if_needed=download_if_needed,
    load_raw=load_raw,
    clean=clean,
    temporal_split=temporal_split,
    prepare=prepare,
    main=main,
)

### Перевірка / демо

In [ ]:
b = data_prep.prepare()
print('train:', len(b['X_train']), '| test:', len(b['X_test']))
print('pos rate train/test:', round(b['y_train'].mean(),3), round(b['y_test'].mean(),3))